## Import packages

In [1]:
from pathlib import Path
from datetime import date, timedelta
import csv
import time
import requests

## Download configuration

In [2]:
BASE_URL = (
    "https://www.consumerfinance.gov/"
    "data-research/consumer-complaints/search/api/v1/"
)

# The script will create this folder inside Downloads.
OUTPUT_DIR = Path.home() / "Downloads" / "CFPB_Banking_Project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2021, 2026)

PRODUCT_GROUPS = {
    "checking_savings": [
        "Checking or savings account"
    ]
}

# You already downloaded these two combinations.
SKIP_COMBINATIONS = {
    ("checking_savings", 2024),
    ("checking_savings", 2025)
}

MAX_EXPORT_ROWS = 100_000
REQUEST_DELAY = 1

## API request and retry functions

In [3]:
session = requests.Session()

session.headers.update({
    "User-Agent": "CFPB-banking-portfolio-project/1.0"
})


def make_request(params, accept="application/json", stream=False):
    """
    Send an API request with retries for temporary errors
    and rate limiting.
    """
    
    headers = {"Accept": accept}
    max_attempts = 6

    for attempt in range(max_attempts):
        try:
            response = session.get(
                BASE_URL,
                params=params,
                headers=headers,
                timeout=(30, 300),
                stream=stream
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                
                if retry_after and retry_after.isdigit():
                    wait_time = int(retry_after)
                else:
                    wait_time = 5 * (2 ** attempt)

                print(
                    f"Rate limited. Waiting {wait_time} seconds..."
                )
                response.close()
                time.sleep(wait_time)
                continue

            if response.status_code in {500, 502, 503, 504}:
                wait_time = 5 * (2 ** attempt)
                
                print(
                    f"Temporary server error "
                    f"{response.status_code}. "
                    f"Waiting {wait_time} seconds..."
                )
                
                response.close()
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response

        except requests.RequestException as error:
            if attempt == max_attempts - 1:
                raise

            wait_time = 5 * (2 ** attempt)
            print(f"Request error: {error}")
            print(f"Waiting {wait_time} seconds before retrying...")
            time.sleep(wait_time)

    raise RuntimeError("The request failed after all retry attempts.")

## Count complaints before downloading

In [4]:
def build_params(products, start_date, end_date):
    """
    Create API parameters.

    end_date is exclusive:
    2026-01-01 means through December 31, 2025.
    """
    
    params = [
        ("date_received_min", start_date.isoformat()),
        ("date_received_max", (end_date - timedelta(days=1)).isoformat())
    ]

    for product in products:
        params.append(("product", product))

    return params


def get_complaint_count(products, start_date, end_date):
    """
    Get the number of matching complaints without
    downloading the entire dataset.
    """
    
    params = build_params(products, start_date, end_date)
    params.extend([
        ("size", 1),
        ("no_aggs", "true"),
        ("no_highlight", "true")
    ])

    response = make_request(params)
    payload = response.json()

    total = payload["hits"]["total"]

    if isinstance(total, dict):
        count = total["value"]
    else:
        count = total

    response.close()
    time.sleep(REQUEST_DELAY)

    return count

## Download and validate each CSV

In [5]:
def count_csv_rows(file_path):
    """
    Count data rows correctly, even if a CSV field contains
    commas or line breaks.
    """
    
    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.reader(file)
        next(reader, None)  # Skip the header
        return sum(1 for _ in reader)


def create_filename(group_name, start_date, end_date):
    """
    Use a short annual filename for full years and a detailed
    filename for split periods.
    """
    
    if (
        start_date.month == 1
        and start_date.day == 1
        and end_date == date(start_date.year + 1, 1, 1)
    ):
        return f"{group_name}_{start_date.year}.csv"

    inclusive_end = end_date - timedelta(days=1)

    return (
        f"{group_name}_"
        f"{start_date.isoformat()}_to_"
        f"{inclusive_end.isoformat()}.csv"
    )


def download_csv(
    group_name,
    products,
    start_date,
    end_date,
    expected_count
):
    """
    Download one filtered CSV and verify its row count.
    """
    
    filename = create_filename(
        group_name,
        start_date,
        end_date
    )

    output_path = OUTPUT_DIR / filename
    temporary_path = output_path.with_suffix(".csv.part")

    if output_path.exists():
        print(f"Already exists — skipping: {filename}")
        return

    params = build_params(products, start_date, end_date)
    params.append(("format", "csv"))

    print(
        f"Downloading {filename}: "
        f"{expected_count:,} complaints"
    )

    response = make_request(
        params,
        accept="text/csv",
        stream=True
    )

    with open(temporary_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    response.close()
    temporary_path.replace(output_path)

    actual_count = count_csv_rows(output_path)

    if actual_count == expected_count:
        print(f"Saved and verified: {actual_count:,} rows\n")
    else:
        print(
            f"Warning: expected {expected_count:,} rows, "
            f"but downloaded {actual_count:,} rows.\n"
        )

    time.sleep(REQUEST_DELAY)

## Automatically split results exceeding 100,000

In [6]:
def extract_date_range(
    group_name,
    products,
    start_date,
    end_date
):
    """
    Download a date range. If it exceeds the CFPB export
    limit, divide the period into smaller ranges automatically.
    """
    
    count = get_complaint_count(
        products,
        start_date,
        end_date
    )

    print(
        f"{group_name} | "
        f"{start_date} to {end_date - timedelta(days=1)}: "
        f"{count:,} complaints"
    )

    if count == 0:
        print("No matching complaints — skipping.\n")
        return

    if count <= MAX_EXPORT_ROWS:
        download_csv(
            group_name,
            products,
            start_date,
            end_date,
            count
        )
        return

    number_of_days = (end_date - start_date).days
    midpoint = start_date + timedelta(days=number_of_days // 2)

    print(
        f"Result exceeds {MAX_EXPORT_ROWS:,}. "
        f"Splitting at {midpoint}.\n"
    )

    extract_date_range(
        group_name,
        products,
        start_date,
        midpoint
    )

    extract_date_range(
        group_name,
        products,
        midpoint,
        end_date
    )

## Confirm the output path

In [7]:
print(OUTPUT_DIR)

~\Downloads\CFPB_Banking_Project


## Test

In [8]:
test_count = get_complaint_count(
    ["Mortgage"],
    date(2021, 1, 1),
    date(2022, 1, 1)
)

print(f"Test count: {test_count:,}")

Test count: 26,547


## Run the extraction

In [9]:
for group_name, products in PRODUCT_GROUPS.items():
    for year in YEARS:
        
        if (group_name, year) in SKIP_COMBINATIONS:
            print(
                f"Already downloaded manually — skipping: "
                f"{group_name} {year}\n"
            )
            continue

        start_date = date(year, 1, 1)
        end_date = date(year + 1, 1, 1)

        extract_date_range(
            group_name,
            products,
            start_date,
            end_date
        )

print("Extraction complete.")
print(f"Files saved in: {OUTPUT_DIR}")

## New extraction cell due to server error. The problem likely occurred when the script began searching three card product names simultaneously.

In [10]:
REMAINING_DOWNLOADS = [
    # Historical combined card category
    (
        "cards_legacy",
        ["Credit card or prepaid card"],
        range(2021, 2024)
    ),

    # Current separate categories, introduced during 2023
    (
        "credit_card",
        ["Credit card"],
        range(2023, 2026)
    ),
    (
        "prepaid_card",
        ["Prepaid card"],
        range(2023, 2026)
    ),

    # Other selected banking products
    (
        "money_transfers",
        ["Money transfer, virtual currency, or money service"],
        range(2021, 2026)
    ),
    (
        "mortgage",
        ["Mortgage"],
        range(2021, 2026)
    )
]


for group_name, products, years in REMAINING_DOWNLOADS:
    for year in years:
        start_date = date(year, 1, 1)
        end_date = date(year + 1, 1, 1)

        extract_date_range(
            group_name,
            products,
            start_date,
            end_date
        )

print("Remaining extraction complete.")
print(f"Files saved in: {OUTPUT_DIR}")

cards_legacy | 2021-01-01 to 2021-12-31: 31,854 complaints
Saved and verified: 31,854 rows

cards_legacy | 2022-01-01 to 2022-12-31: 39,921 complaints
Saved and verified: 39,921 rows

cards_legacy | 2023-01-01 to 2023-12-31: 35,310 complaints
Saved and verified: 35,310 rows

credit_card | 2023-01-01 to 2023-12-31: 21,948 complaints
Saved and verified: 21,948 rows

credit_card | 2024-01-01 to 2024-12-31: 76,109 complaints
Saved and verified: 76,109 rows

credit_card | 2025-01-01 to 2025-12-31: 90,157 complaints
Saved and verified: 90,157 rows

prepaid_card | 2023-01-01 to 2023-12-31: 2,281 complaints
Saved and verified: 2,281 rows

prepaid_card | 2024-01-01 to 2024-12-31: 6,341 complaints
Saved and verified: 6,341 rows

Temporary server error 504. Waiting 5 seconds...
prepaid_card | 2025-01-01 to 2025-12-31: 6,516 complaints
Saved and verified: 6,516 rows

money_transfers | 2021-01-01 to 2021-12-31: 13,906 complaints
Saved and verified: 13,906 rows

money_transfers | 2022-01-01 to 2022-

## Add vehicle loans

In [11]:
for year in range(2021, 2026):
    start_date = date(year, 1, 1)
    end_date = date(year + 1, 1, 1)

    extract_date_range(
        "vehicle_loans",
        ["Vehicle loan or lease"],
        start_date,
        end_date
    )

print("Vehicle loan extraction complete.")

vehicle_loans | 2021-01-01 to 2021-12-31: 7,827 complaints
Saved and verified: 7,827 rows

vehicle_loans | 2022-01-01 to 2022-12-31: 8,654 complaints
Saved and verified: 8,654 rows

vehicle_loans | 2023-01-01 to 2023-12-31: 12,806 complaints
Saved and verified: 12,806 rows

vehicle_loans | 2024-01-01 to 2024-12-31: 13,378 complaints
Saved and verified: 13,378 rows

vehicle_loans | 2025-01-01 to 2025-12-31: 20,823 complaints
Saved and verified: 20,823 rows

Vehicle loan extraction complete.


## Check that everything is complete

In [12]:
csv_files = sorted(OUTPUT_DIR.glob("*.csv"))

total_rows = 0

for file_path in csv_files:
    rows = count_csv_rows(file_path)
    total_rows += rows
    print(f"{file_path.name}: {rows:,} rows")

print("-" * 50)
print(f"Number of CSV files: {len(csv_files)}")
print(f"Combined number of rows: {total_rows:,}")

cards_legacy_2021.csv: 31,854 rows
cards_legacy_2022.csv: 39,921 rows
cards_legacy_2023.csv: 35,310 rows
checking_savings_2021.csv: 29,588 rows
checking_savings_2022.csv: 37,617 rows
checking_savings_2023.csv: 49,917 rows
checking_savings_2024.csv: 52,821 rows
checking_savings_2025.csv: 84,195 rows
credit_card_2023.csv: 21,948 rows
credit_card_2024.csv: 76,109 rows
credit_card_2025.csv: 90,157 rows
money_transfers_2021.csv: 13,906 rows
money_transfers_2022.csv: 13,551 rows
money_transfers_2023.csv: 13,764 rows
money_transfers_2024.csv: 16,772 rows
money_transfers_2025.csv: 84,708 rows
mortgage_2021.csv: 26,547 rows
mortgage_2022.csv: 23,290 rows
mortgage_2023.csv: 22,869 rows
mortgage_2024.csv: 21,473 rows
mortgage_2025.csv: 24,735 rows
prepaid_card_2023.csv: 2,281 rows
prepaid_card_2024.csv: 6,341 rows
prepaid_card_2025.csv: 6,516 rows
vehicle_loans_2021.csv: 7,827 rows
vehicle_loans_2022.csv: 8,654 rows
vehicle_loans_2023.csv: 12,806 rows
vehicle_loans_2024.csv: 13,378 rows
vehicle_l

## Inspect the column names

In [13]:
import pandas as pd

sample_file = OUTPUT_DIR / "checking_savings_2021.csv"

sample = pd.read_csv(sample_file, nrows=5)

print("Columns:")
for column in sample.columns:
    print(repr(column))

display(sample)
sample.info()

Columns:
'Date received'
'Product'
'Sub-product'
'Issue'
'Sub-issue'
'Company public response'
'Company'
'State'
'ZIP code'
'Tags'
'Submitted via'
'Date sent to company'
'Company response to consumer'
'Timely response?'
'Complaint ID'


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2021-06-26T04:57:05.000Z,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,Company has responded to the consumer and the ...,U.S. BANCORP,CA,90706,NaN,Referral,2021-06-28T17:00:13.000Z,Closed with monetary relief,Yes,4500008
1,2021-06-26T01:30:27.000Z,Checking or savings account,Checking account,Opening an account,Unable to open an account,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",MD,20874,NaN,Web,2021-06-26T05:46:48.000Z,Closed with explanation,Yes,4494171
2,2021-06-26T03:53:32.000Z,Checking or savings account,Checking account,Closing an account,Company closed your account,NaN,"AXOS FINANCIAL, INC.",OH,44203,NaN,Web,2021-06-28T20:01:09.000Z,Closed with explanation,Yes,4495300
3,2021-06-26T04:57:22.000Z,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",NaN,XXXXX,NaN,Referral,2021-06-28T14:34:54.000Z,Closed with explanation,Yes,4496952
4,2021-05-11T21:55:21.000Z,Checking or savings account,Checking account,Managing an account,Problem accessing account,NaN,BBVA FINANCIAL CORPORATION,CA,94566,NaN,Web,2021-05-11T22:06:01.000Z,Closed with explanation,Yes,4369852


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Date received                 5 non-null      str    
 1   Product                       5 non-null      str    
 2   Sub-product                   5 non-null      str    
 3   Issue                         5 non-null      str    
 4   Sub-issue                     5 non-null      str    
 5   Company public response       3 non-null      str    
 6   Company                       5 non-null      str    
 7   State                         4 non-null      str    
 8   ZIP code                      5 non-null      str    
 9   Tags                          0 non-null      float64
 10  Submitted via                 5 non-null      str    
 11  Date sent to company          5 non-null      str    
 12  Company response to consumer  5 non-null      str    
 13  Timely response?    